In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import bitsandbytes as bnb

from PIL import Image
from torchvision import transforms

from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler

from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed


# ================= CONFIG =================
MODEL_ID = "runwayml/stable-diffusion-v1-5"

DATA_DIR = "/kaggle/input/datasets/awaiskabir/training-dataset/training_dataset"
OUTPUT_DIR = "/kaggle/working/cub_dreambooth_output"

RESOLUTION = 384
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4

LR = 5e-6
MAX_STEPS = 8195
WARMUP_STEPS = 500
SAVE_EVERY = 2000

MIXED_PRECISION = "fp16"
SEED = 42
# ==========================================


class CUBDreamBoothDataset(Dataset):
    IMAGE_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")

    def __init__(self, data_dir, tokenizer, resolution):
        self.tokenizer = tokenizer
        self.image_paths = []
        self.captions = []

        data_root = Path(data_dir)

        if not data_root.exists():
            raise FileNotFoundError(f"Dataset path not found: {data_root}")

        for class_dir in sorted(data_root.iterdir()):
            if not class_dir.is_dir():
                continue

            for ext in self.IMAGE_EXTS:
                for img_path in class_dir.glob(ext):
                    txt_path = img_path.with_suffix(".txt")

                    if txt_path.exists():
                        caption = txt_path.read_text(encoding="utf-8").strip()
                    else:
                        species = class_dir.name
                        species = species.replace("_", " ")
                        species = species.replace("-", " ")
                        species = species.replace(".", " ")
                        caption = f"a photo of a {species} bird"

                    if caption:
                        self.image_paths.append(img_path)
                        self.captions.append(caption)

        if len(self.image_paths) == 0:
            raise RuntimeError("No images found. Check DATA_DIR.")

        print(f"[Dataset] Found {len(self.image_paths)} images.")

        self.transform = transforms.Compose([
            transforms.Resize(
                resolution,
                interpolation=transforms.InterpolationMode.BILINEAR
            ),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        pixel_values = self.transform(image)

        input_ids = self.tokenizer(
            self.captions[idx],
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids[0]

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
        }


def main():
    gc.collect()
    torch.cuda.empty_cache()

    set_seed(SEED)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    accelerator = Accelerator(
        mixed_precision=MIXED_PRECISION,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        project_dir=OUTPUT_DIR,
    )

    tokenizer = CLIPTokenizer.from_pretrained(
        MODEL_ID,
        subfolder="tokenizer"
    )

    text_encoder = CLIPTextModel.from_pretrained(
        MODEL_ID,
        subfolder="text_encoder"
    )

    vae = AutoencoderKL.from_pretrained(
        MODEL_ID,
        subfolder="vae"
    )

    unet = UNet2DConditionModel.from_pretrained(
        MODEL_ID,
        subfolder="unet"
    )

    noise_scheduler = DDPMScheduler.from_pretrained(
        MODEL_ID,
        subfolder="scheduler"
    )

    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)

    vae.to(accelerator.device, dtype=torch.float16)
    text_encoder.to(accelerator.device, dtype=torch.float16)

    unet.train()
    unet.enable_gradient_checkpointing()

    vae.enable_slicing()
    vae.enable_tiling()

    try:
        unet.enable_xformers_memory_efficient_attention()
        print("xFormers enabled.")
    except Exception:
        print("xFormers not available. Continuing without it.")

    dataset = CUBDreamBoothDataset(
        DATA_DIR,
        tokenizer,
        RESOLUTION
    )

    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
    )

    print(f"Dataset size: {len(dataset)}")
    print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

    optimizer = bnb.optim.AdamW8bit(
        unet.parameters(),
        lr=LR,
        betas=(0.9, 0.999),
        weight_decay=1e-2,
        eps=1e-8,
    )

    lr_scheduler = get_scheduler(
        "cosine_with_restarts",
        optimizer=optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=MAX_STEPS,
        num_cycles=3,
    )

    unet, optimizer, dataloader, lr_scheduler = accelerator.prepare(
        unet,
        optimizer,
        dataloader,
        lr_scheduler,
    )

    global_step = 0

    print("Starting DreamBooth-style fine-tuning...")

    while global_step < MAX_STEPS:
        for batch in dataloader:
            with accelerator.accumulate(unet):

                pixel_values = batch["pixel_values"].to(
                    accelerator.device,
                    dtype=torch.float16
                )

                input_ids = batch["input_ids"].to(accelerator.device)

                with torch.no_grad():
                    latents = vae.encode(pixel_values).latent_dist.sample()
                    latents = latents * vae.config.scaling_factor

                    encoder_hidden_states = text_encoder(input_ids)[0]

                noise = torch.randn_like(latents)

                timesteps = torch.randint(
                    0,
                    noise_scheduler.config.num_train_timesteps,
                    (latents.shape[0],),
                    device=latents.device,
                ).long()

                noisy_latents = noise_scheduler.add_noise(
                    latents,
                    noise,
                    timesteps
                )

                noise_pred = unet(
                    noisy_latents,
                    timesteps,
                    encoder_hidden_states
                ).sample

                loss = F.mse_loss(
                    noise_pred.float(),
                    noise.float(),
                    reduction="mean"
                )

                accelerator.backward(loss)

                # Do NOT use accelerator.clip_grad_norm_ here.
                # It causes "Attempting to unscale FP16 gradients" with fp16 + AdamW8bit.

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1

                if global_step % 100 == 0:
                    print(
                        f"Step {global_step}/{MAX_STEPS} | "
                        f"Loss: {loss.detach().item():.4f} | "
                        f"LR: {lr_scheduler.get_last_lr()[0]:.2e}"
                    )

                if global_step % SAVE_EVERY == 0:
                    ckpt_path = f"{OUTPUT_DIR}/checkpoint-{global_step}"
                    accelerator.save_state(ckpt_path)
                    print(f"Checkpoint saved: {ckpt_path}")

            if global_step >= MAX_STEPS:
                break

    accelerator.wait_for_everyone()

    if accelerator.is_main_process:
        unwrapped_unet = accelerator.unwrap_model(unet)

        pipe = StableDiffusionPipeline.from_pretrained(
            MODEL_ID,
            unet=unwrapped_unet,
            torch_dtype=torch.float16,
            safety_checker=None,
        )

        pipe.save_pretrained(OUTPUT_DIR)

        print(f"Fine-tuned DreamBooth model saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'bitsandbytes'

In [ ]:
import shutil
import os

OUTPUT_DIR = "/kaggle/working/cub_dreambooth_output"

for ckpt in ["checkpoint-2000", "checkpoint-4000", "checkpoint-6000"]:
    path = os.path.join(OUTPUT_DIR, ckpt)
    
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Deleted:", ckpt)

In [ ]:
import os

ckpt = "/kaggle/working/cub_dreambooth_output/checkpoint-8000"

for root, dirs, files in os.walk(ckpt):
    print(root)
    for f in files[:5]:
        print("   ", f)

In [ ]:
from diffusers import StableDiffusionPipeline, UNet2DConditionModel
from safetensors.torch import load_file
import torch

MODEL_ID = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = "/kaggle/working/cub_dreambooth_output"
CKPT = f"{OUTPUT_DIR}/checkpoint-8000/model.safetensors"

# 1. Base UNet architecture load karo
unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID,
    subfolder="unet",
    torch_dtype=torch.float16,
)

# 2. Trained weights load karo
state_dict = load_file(CKPT)
unet.load_state_dict(state_dict)

# 3. Final pipeline banao
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    unet=unet,
    torch_dtype=torch.float16,
    safety_checker=None,
)

# 4. Save final model
pipe.save_pretrained(f"{OUTPUT_DIR}/final_model")

print("FINAL MODEL SAVED SUCCESSFULLY")

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/cub_dreambooth_output"

for x in os.listdir(OUTPUT_DIR):
    print(x)

In [ ]:
!pip install bitsandbytes -q

In [ ]:
!accelerate launch train_cub_dreambooth.py

# Inference

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import os
import matplotlib.pyplot as plt

# Create output directory
os.makedirs("/kaggle/working/outputs", exist_ok=True)

# Load DreamBooth model
pipe = StableDiffusionPipeline.from_pretrained(
    "/kaggle/working/cub_dreambooth_output/final_model",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

# Negative prompt
negative_prompt = (
    "cartoon, anime, illustration, painting, drawing, cgi, 3d render, "
    "low quality, blurry, bad anatomy, deformed bird, mutated bird, "
    "extra wings, extra legs, extra beak, two heads, wrong proportions, "
    "text, watermark, logo"
)

# Bird prompts
species_prompts = [
    "ultra realistic wildlife photograph of a black footed albatross standing on a rocky ocean shore, curved pale bill, dark wings, white breast, natural sunlight, sharp feathers, high detail, DSLR photo, 85mm lens",

    "ultra realistic wildlife photograph of a painted bunting perched on a thin branch, bright red breast, green upperparts, blue crown, detailed feathers, soft forest background, natural lighting, DSLR photo, high resolution",

    "ultra realistic wildlife photograph of a baltimore oriole sitting on a tree branch, vivid orange underparts, black head, white wing bars, sharp beak, blurred green background, natural sunlight, high detail photo",

    "ultra realistic wildlife photograph of a blue jay perched on a wooden fence, bright blue feathers, white chest, black collar, detailed wings, natural outdoor lighting, shallow depth of field, DSLR photo",

    "ultra realistic wildlife photograph of a northern cardinal perched on a snowy branch, vivid red feathers, black face mask, pointed crest, natural winter background, sharp focus, high detail DSLR photo",

    "ultra realistic wildlife photograph of an american goldfinch perched on sunflower stem, bright yellow body, black wings, small orange beak, natural garden background, soft sunlight, detailed feathers",

    "ultra realistic wildlife photograph of a scarlet tanager perched in green forest, scarlet red body, black wings, short bill, realistic feathers, natural light, blurred background, high resolution photo",

    "ultra realistic wildlife photograph of a cedar waxwing sitting on berry branch, smooth brown plumage, yellow tail tip, black mask, detailed feathers, soft natural lighting, realistic bird photography",

    "ultra realistic wildlife photograph of a ruby throated hummingbird hovering near flowers, iridescent green feathers, red throat, tiny fast wings, natural garden background, sharp focus, high detail",

    "ultra realistic wildlife photograph of a snowy owl standing in snow, white plumage, yellow eyes, black speckles, realistic feathers, cold arctic background, natural light, sharp DSLR photo",

    "ultra realistic wildlife photograph of a peregrine falcon perched on cliff edge, gray wings, barred chest, hooked beak, intense eyes, realistic feather texture, dramatic natural lighting, high detail",

    "ultra realistic wildlife photograph of a great blue heron standing in shallow water, long legs, blue gray feathers, sharp beak, wetland background, natural sunlight, realistic wildlife photo",

    "ultra realistic wildlife photograph of a kingfisher perched above a river, blue back, orange belly, long pointed beak, detailed feathers, water background, shallow depth of field, DSLR photo",

    "ultra realistic wildlife photograph of a flamingo standing in a lake, pink feathers, curved neck, long thin legs, natural water reflection, realistic lighting, high resolution wildlife photo",

    "ultra realistic wildlife photograph of a peacock displaying colorful tail feathers, blue neck, elegant crest, detailed plumage, natural garden background, sharp focus, DSLR photo"
]

# Generate and display images
for idx, prompt in enumerate(species_prompts):

    print(f"\nGenerating image {idx+1}/{len(species_prompts)}")
    print(f"Prompt: {prompt}")

    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
    ).images[0]

    # Create filename
    fname = (
        prompt[:50]
        .replace(" ", "_")
        .replace(",", "")
        .replace("/", "_")
        + ".png"
    )

    # Save image
    save_path = f"/kaggle/working/outputs/{fname}"
    image.save(save_path)

    print(f"Saved: {save_path}")

    # Display image
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Bird {idx+1}")
    plt.show()

print("All bird images generated, saved, and displayed successfully!")

In [ ]:
from PIL import Image
from IPython.display import display
import os

OUTPUTS = "/kaggle/working/outputs"

for fname in os.listdir(OUTPUTS):
    if fname.endswith(".png"):
        print(fname)
        img = Image.open(os.path.join(OUTPUTS, fname))
        display(img)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import os
import matplotlib.pyplot as plt

# Load Stable Diffusion model
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

# Create output directory
os.makedirs("/kaggle/working/base_outputs", exist_ok=True)

# Negative prompt
negative_prompt = (
    "cartoon, anime, illustration, painting, drawing, cgi, 3d render, "
    "low quality, blurry, bad anatomy, deformed bird, mutated bird, "
    "extra wings, extra legs, extra beak, two heads, wrong proportions, "
    "text, watermark, logo"
)

# Bird prompts
species_prompts = [
    "a photo of a black footed albatross, with curved bill, black wings, white breast",
    "a photo of a painted bunting, with red breast, green upperparts, blue crown",
    "a photo of a baltimore oriole, with orange underparts, black head, white wing bars",
    "a photo of a blue jay, with bright blue feathers, white chest, black collar",
    "a photo of a northern cardinal, with vivid red feathers, black face mask, pointed crest",
    "a photo of an american goldfinch, with yellow body, black wings, small orange beak",
    "a photo of a scarlet tanager, with scarlet red body, black wings, short bill",
    "a photo of a cedar waxwing, with smooth brown plumage, yellow tail tip, black mask",
    "a photo of a ruby throated hummingbird, with iridescent green feathers, red throat, tiny wings",
    "a photo of a snowy owl, with white plumage, yellow eyes, black speckles",
    "a photo of a peregrine falcon, with gray wings, barred chest, hooked beak",
    "a photo of a great blue heron, with long legs, blue gray feathers, sharp beak",
    "a photo of a kingfisher, with blue back, orange belly, long pointed beak",
    "a photo of a flamingo, with pink feathers, curved neck, long thin legs",
    "a photo of a peacock, with colorful tail feathers, blue neck, elegant crest",
]

# Generate images
for prompt in species_prompts:

    print("\nPrompt:", prompt)

    images = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        num_images_per_prompt=4,
    ).images

    # Display images in a row
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    for i, img in enumerate(images):

        # Create filename
        fname = (
            prompt[:50]
            .replace(" ", "_")
            .replace(",", "")
            .replace("/", "_")
            + f"_{i}.png"
        )

        save_path = f"/kaggle/working/base_outputs/{fname}"

        # Save image
        img.save(save_path)
        print("Saved:", save_path)

        # Display image
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"Image {i+1}")

    plt.tight_layout()
    plt.show()

print("All bird images generated, saved, and displayed successfully!")

# Base Inference

In [ ]:
!pip install torchmetrics[image] torch-fidelity open-clip-torch -q

In [ ]:
REAL_DIR = "/kaggle/input/training_dataset/training_dataset"
BASE_DIR = "/kaggle/working/base_outputs"
DREAM_DIR = "/kaggle/working/outputs"

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

REAL_DIR = "/kaggle/input/datasets/awaiskabir/training-dataset/training_dataset"
BASE_DIR = "/kaggle/working/base_outputs"
DREAM_DIR = "/kaggle/working/outputs"

species_prompts = [
    "a photo of a black footed albatross, with curved bill, black wings, white breast",
    "a photo of a painted bunting, with red breast, green upperparts, blue crown",
    "a photo of a baltimore oriole, with orange underparts, black head, white wing bars",
]

def load_images(folder, max_images=None):
    imgs = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(root, f)
                imgs.append(Image.open(path).convert("RGB"))
                if max_images and len(imgs) >= max_images:
                    return imgs
    return imgs

# Real images bohat zyada ho sakti hain, speed ke liye 500 use kar rahe hain
real_imgs = load_images(REAL_DIR, max_images=500)
base_imgs = load_images(BASE_DIR)
dream_imgs = load_images(DREAM_DIR)

print("Real Images:", len(real_imgs))
print("Base Images:", len(base_imgs))
print("DreamBooth Images:", len(dream_imgs))

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import os

# Create output directory
os.makedirs("/kaggle/working/outputs", exist_ok=True)

# Load trained DreamBooth model
pipe = StableDiffusionPipeline.from_pretrained(
    "/kaggle/working/cub_dreambooth_output/final_model",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

# 30 different bird prompts
species_prompts = [
    "a photo of a black footed albatross, with curved bill, black wings, white breast",
    "a photo of a painted bunting, with red breast, green upperparts, blue crown",
    "a photo of a baltimore oriole, with orange underparts, black head, white wing bars",
    "a photo of a blue jay, with bright blue feathers, white chest, black collar",
    "a photo of a northern cardinal, with vivid red feathers, black face mask, pointed crest",
    "a photo of an american goldfinch, with yellow body, black wings, small orange beak",
    "a photo of a scarlet tanager, with scarlet red body, black wings, short bill",
    "a photo of a cedar waxwing, with smooth brown plumage, yellow tail tip, black mask",
    "a photo of a ruby throated hummingbird, with iridescent green feathers, red throat, tiny wings",
    "a photo of a snowy owl, with white plumage, yellow eyes, black speckles",
    "a photo of a peregrine falcon, with gray wings, barred chest, hooked beak",
    "a photo of a great blue heron, with long legs, blue gray feathers, sharp beak",
    "a photo of a kingfisher, with blue back, orange belly, long pointed beak",
    "a photo of a flamingo, with pink feathers, curved neck, long thin legs",
    "a photo of a peacock, with colorful tail feathers, blue neck, elegant crest",
    "a photo of a toucan, with large colorful bill, black body, yellow throat",
    "a photo of a macaw parrot, with vibrant blue and yellow feathers, curved beak",
    "a photo of a bald eagle, with white head, dark brown body, yellow hooked beak",
    "a photo of a puffin, with black and white feathers, colorful beak, orange feet",
    "a photo of a robin, with orange breast, gray wings, small yellow beak",
    "a photo of a barn owl, with heart shaped face, pale feathers, dark eyes",
    "a photo of a woodpecker, with red crest, black and white wings, sharp beak",
    "a photo of a swan, with pure white feathers, long neck, orange beak",
    "a photo of a mallard duck, with green head, brown chest, yellow bill",
    "a photo of a cockatoo, with white feathers, yellow crest, curved gray beak",
    "a photo of a canary, with bright yellow feathers, tiny orange beak",
    "a photo of an ostrich, with long neck, black and white feathers, strong legs",
    "a photo of an emperor penguin, with black back, white belly, yellow neck patches",
    "a photo of a hornbill, with large curved bill, black plumage, white tail",
    "a photo of a red winged blackbird, with glossy black feathers, red shoulder patches"
]

# Generate images
for prompt in species_prompts:
    images = pipe(
        prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        num_images_per_prompt=1,
    ).images

    for i, img in enumerate(images):
        fname = (
            prompt[:50]
            .replace(" ", "_")
            .replace(",", "")
            .replace("/", "_")
            + f"_{i}.png"
        )

        img.save(f"/kaggle/working/outputs/{fname}")
        print(f"Saved: {fname}")

print("All bird images generated successfully!")

In [ ]:
from PIL import Image
from IPython.display import display
import os

OUTPUTS = "/kaggle/working/outputs"

for fname in os.listdir(OUTPUTS):
    if fname.endswith(".png"):
        print(fname)
        img = Image.open(os.path.join(OUTPUTS, fname))
        display(img)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import os
import matplotlib.pyplot as plt

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

os.makedirs("/kaggle/working/base_outputs", exist_ok=True)

# 15 attribute-based bird prompts
species_prompts = [

    "a realistic photo of a black footed albatross bird, curved bill, black wings, white breast",

    "a realistic photo of a painted bunting bird, red breast, green upperparts, blue crown",

    "a realistic photo of a baltimore oriole bird, orange underparts, black head, white wing bars",

    "a realistic photo of a blue jay bird, bright blue feathers, white chest, black collar",

    "a realistic photo of a northern cardinal bird, vivid red feathers, black face mask, pointed crest",

    "a realistic photo of an american goldfinch bird, yellow body, black wings, small orange beak",

    "a realistic photo of a scarlet tanager bird, scarlet red body, black wings, short bill",

    "a realistic photo of a cedar waxwing bird, smooth brown plumage, yellow tail tip, black mask",

    "a realistic photo of a ruby throated hummingbird, iridescent green feathers, red throat, tiny wings",

    "a realistic photo of a snowy owl bird, white plumage, yellow eyes, black speckles",

    "a realistic photo of a peregrine falcon bird, gray wings, barred chest, hooked beak",

    "a realistic photo of a great blue heron bird, long legs, blue gray feathers, sharp beak",

    "a realistic photo of a kingfisher bird, blue back, orange belly, long pointed beak",

    "a realistic photo of a flamingo bird, pink feathers, curved neck, long thin legs",

    "a realistic photo of a peacock bird, colorful tail feathers, blue neck, elegant crest"
]

for prompt_idx, prompt in enumerate(species_prompts):

    print("\nPrompt:", prompt)

    images = pipe(
        prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        num_images_per_prompt=1,
    ).images

    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))

    if len(images) == 1:
        axes = [axes]

    for i, img in enumerate(images):

        fname = (
            f"base_{prompt_idx+1:02d}_"
            + prompt[:45]
            .replace(" ", "_")
            .replace(",", "")
            .replace("/", "_")
            + f"_{i}.png"
        )

        save_path = f"/kaggle/working/base_outputs/{fname}"

        img.save(save_path)

        print("Saved:", save_path)

        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"Image {i+1}")

    plt.tight_layout()
    plt.show()

print("All 15 bird images generated successfully!")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

REAL_DIR = "/kaggle/input/datasets/awaiskabir/training-dataset/training_dataset"
BASE_DIR = "/kaggle/working/base_outputs"
DREAM_DIR = "/kaggle/working/outputs"

species_prompts = [
    "a photo of a black footed albatross, with curved bill, black wings, white breast",
    "a photo of a painted bunting, with red breast, green upperparts, blue crown",
    "a photo of a baltimore oriole, with orange underparts, black head, white wing bars",
]

def load_images(folder, max_images=None):
    imgs = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(root, f)
                imgs.append(Image.open(path).convert("RGB"))
                if max_images and len(imgs) >= max_images:
                    return imgs
    return imgs

# Real images bohat zyada ho sakti hain, speed ke liye 500 use kar rahe hain
real_imgs = load_images(REAL_DIR, max_images=500)
base_imgs = load_images(BASE_DIR)
dream_imgs = load_images(DREAM_DIR)

print("Real Images:", len(real_imgs))
print("Base Images:", len(base_imgs))
print("DreamBooth Images:", len(dream_imgs))

In [ ]:
min_count = min(len(base_imgs), len(dream_imgs))

base_imgs = base_imgs[:min_count]
dream_imgs = dream_imgs[:min_count]

print("Using", min_count, "images for fair comparison")

In [ ]:
fid_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 255).byte())
])

def calculate_fid(real_images, gen_images):
    fid = FrechetInceptionDistance(feature=2048).to(device)

    for img in real_images:
        x = fid_transform(img).unsqueeze(0).to(device)
        fid.update(x, real=True)

    for img in gen_images:
        x = fid_transform(img).unsqueeze(0).to(device)
        fid.update(x, real=False)

    return fid.compute().item()

base_fid = calculate_fid(real_imgs, base_imgs)
dream_fid = calculate_fid(real_imgs, dream_imgs)

print("Base FID:", base_fid)
print("DreamBooth FID:", dream_fid)

In [ ]:
isc_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 255).byte())
])

def calculate_isc(images):
    isc = InceptionScore().to(device)

    for img in images:
        x = isc_transform(img).unsqueeze(0).to(device)
        isc.update(x)

    mean, std = isc.compute()
    return mean.item(), std.item()

base_isc, base_isc_std = calculate_isc(base_imgs)
dream_isc, dream_isc_std = calculate_isc(dream_imgs)

print("Base ISC:", base_isc, "+/-", base_isc_std)
print("DreamBooth ISC:", dream_isc, "+/-", dream_isc_std)

In [ ]:
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

clip_model = clip_model.to(device)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

def calculate_clip_score(images, prompts):
    scores = []

    with torch.no_grad():
        for i, img in enumerate(images):
            prompt = prompts[i % len(prompts)]

            image_tensor = preprocess(img).unsqueeze(0).to(device)
            text_tensor = tokenizer([prompt]).to(device)

            image_features = clip_model.encode_image(image_tensor)
            text_features = clip_model.encode_text(text_tensor)

            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            score = (image_features @ text_features.T).item()
            scores.append(score)

    return float(np.mean(scores))

base_clip = calculate_clip_score(base_imgs, species_prompts)
dream_clip = calculate_clip_score(dream_imgs, species_prompts)

print("Base CLIP Score:", base_clip)
print("DreamBooth CLIP Score:", dream_clip)

In [ ]:
results = pd.DataFrame({
    "Model": ["Base Stable Diffusion", "DreamBooth"],
    "FID ↓": [base_fid, dream_fid],
    "ISC ↑": [base_isc, dream_isc],
    "ISC Std": [base_isc_std, dream_isc_std],
    "CLIP Score ↑": [base_clip, dream_clip],
})

results

In [ ]:
!pip install torchmetrics[image] torch-fidelity open-clip-torch -q

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

# Paths
REAL_DIR = "/kaggle/input/datasets/awaiskabir/training-dataset/training_dataset"
DREAM_DIR = "/kaggle/working/outputs"

species_prompts = [
    "a photo of a black footed albatross, with curved bill, black wings, white breast",
    "a photo of a painted bunting, with red breast, green upperparts, blue crown",
    "a photo of a baltimore oriole, with orange underparts, black head, white wing bars",
]

# ----------------------------
# Load Images
# ----------------------------
def load_images(folder, max_images=None):
    imgs = []

    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):

                path = os.path.join(root, f)

                img = Image.open(path).convert("RGB")

                imgs.append(img)

                if max_images and len(imgs) >= max_images:
                    return imgs

    return imgs

real_imgs = load_images(REAL_DIR, max_images=500)
dream_imgs = load_images(DREAM_DIR)

# Fair comparison
min_count = min(len(real_imgs), len(dream_imgs))

real_imgs = real_imgs[:min_count]
dream_imgs = dream_imgs[:min_count]

print("Using", min_count, "images")

In [ ]:
fid_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 255).byte())
])

def calculate_fid(real_images, gen_images):

    fid = FrechetInceptionDistance(feature=2048).to(device)

    for img in real_images:
        x = fid_transform(img).unsqueeze(0).to(device)
        fid.update(x, real=True)

    for img in gen_images:
        x = fid_transform(img).unsqueeze(0).to(device)
        fid.update(x, real=False)

    return fid.compute().item()

dream_fid = calculate_fid(real_imgs, dream_imgs)

print("DreamBooth FID:", dream_fid)

In [ ]:
isc_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 255).byte())
])

def calculate_isc(images):

    isc = InceptionScore().to(device)

    for img in images:
        x = isc_transform(img).unsqueeze(0).to(device)
        isc.update(x)

    mean, std = isc.compute()

    return mean.item(), std.item()

dream_isc, dream_isc_std = calculate_isc(dream_imgs)

print("DreamBooth ISC:", dream_isc)
print("ISC Std:", dream_isc_std)

In [ ]:
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

clip_model = clip_model.to(device)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

def calculate_clip_score(images, prompts):

    scores = []

    with torch.no_grad():

        for i, img in enumerate(images):

            prompt = prompts[i % len(prompts)]

            image_tensor = preprocess(img).unsqueeze(0).to(device)

            text_tensor = tokenizer([prompt]).to(device)

            image_features = clip_model.encode_image(image_tensor)

            text_features = clip_model.encode_text(text_tensor)

            image_features /= image_features.norm(dim=-1, keepdim=True)

            text_features /= text_features.norm(dim=-1, keepdim=True)

            score = (image_features @ text_features.T).item()

            scores.append(score)

    return float(np.mean(scores))

dream_clip = calculate_clip_score(dream_imgs, species_prompts)

print("DreamBooth CLIP Score:", dream_clip)

In [ ]:
results = pd.DataFrame({
    "Comparison": ["Real Dataset vs DreamBooth"],
    "Real Images Used": [len(real_imgs)],
    "DreamBooth Images Used": [len(dream_imgs)],
    "FID ↓": [dream_fid],
    "DreamBooth ISC ↑": [dream_isc],
    "DreamBooth CLIP Score ↑": [dream_clip],
})

results

In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance

device = "cuda" if torch.cuda.is_available() else "cpu"

REAL_DIR = "/kaggle/input/datasets/awaiskabir/training-dataset/training_dataset"
DREAM_DIR = "/kaggle/working/outputs"

def load_images(folder, max_images=None):
    imgs = []

    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(root, f)
                img = Image.open(path).convert("RGB")
                imgs.append(img)

                if max_images and len(imgs) >= max_images:
                    return imgs

    return imgs

real_imgs = load_images(REAL_DIR, max_images=42)
dream_imgs = load_images(DREAM_DIR)

min_count = min(len(real_imgs), len(dream_imgs))
real_imgs = real_imgs[:min_count]
dream_imgs = dream_imgs[:min_count]

print("Real Images Used:", len(real_imgs))
print("DreamBooth Images Used:", len(dream_imgs))

fid_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 255).byte())
])

fid = FrechetInceptionDistance(feature=2048).to(device)

for img in real_imgs:
    x = fid_transform(img).unsqueeze(0).to(device)
    fid.update(x, real=True)

for img in dream_imgs:
    x = fid_transform(img).unsqueeze(0).to(device)
    fid.update(x, real=False)

dream_fid = fid.compute().item()

print("FID Real Dataset vs DreamBooth:", dream_fid)

if dream_fid < 50:
    print("Result: Very good similarity to real dataset.")
elif dream_fid < 100:
    print("Result: Good similarity to real dataset.")
elif dream_fid < 200:
    print("Result: Moderate similarity to real dataset.")
else:
    print("Result: Weak similarity to real dataset.")